# MapReduce on Local Hadoop — Student Guide

Step-by-step notebook for **creating**, **running**, and **monitoring** MapReduce jobs on the Docker Hadoop cluster.

| | |
|---|---|
| **Scenario** | **ShopStream** e-commerce — analyze product review text |
| **Prerequisite** | Cluster running — [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) Steps 1–4 |
| **Run from** | `hadoop-local-docker/` directory |
| **Written guide** | [MAPREDUCE-STUDENT-GUIDE.md](./MAPREDUCE-STUDENT-GUIDE.md) |

**How to use:** Run cells **in order**. Each step tells you what to expect before you run the next cell.

### Monitoring URLs — open these in your browser

| UI | URL |
|----|-----|
| YARN ResourceManager | http://localhost:8088 |
| YARN Proxy | http://localhost:9099 |
| MapReduce History | http://localhost:19888 |
| HDFS NameNode | http://localhost:9870 |

---
## Step 1 — What is MapReduce?

**Goal:** ShopStream wants to know which words appear most often in customer reviews.

MapReduce splits batch processing into three phases:

```mermaid
flowchart LR
  subgraph input [HDFS Input]
    L1[Review line 1]
    L2[Review line 2]
  end
  subgraph map [Map]
    M[Mapper emits word, 1]
  end
  subgraph shuffle [Shuffle]
    S[Group by word]
  end
  subgraph reduce [Reduce]
    R[Sum counts]
  end
  subgraph output [HDFS Output]
    O[part-r-00000]
  end
  L1 --> M
  L2 --> M
  M --> S --> R --> O
```

| Phase | Your code? | Example output |
|-------|------------|----------------|
| **Map** | Yes | `(delivery, 1)`, `(excellent, 1)` |
| **Shuffle** | No — Hadoop handles it | Groups all `delivery` keys |
| **Reduce** | Yes | `delivery\t2` |

---
## Step 2 — Test the algorithm locally (no Hadoop)

First prove the word-count logic works on your laptop. Hadoop only adds **distribution** across workers.

In [16]:
from pathlib import Path
from mapreduce.local_simulation import run_wordcount, top_n

reviews_path = Path("data/ecommerce/product_reviews.txt")
lines = reviews_path.read_text(encoding="utf-8").splitlines()

print(f"Input: {len(lines)} review lines")
print("--- First 3 lines ---")
for line in lines[:3]:
    print(line)

Input: 15 review lines
--- First 3 lines ---
Great headphones excellent sound quality fast delivery
Headphones broke after one week very disappointed poor quality
Love the cotton t-shirt comfortable fit will buy again


In [17]:
counts = run_wordcount(lines)

print(f"Unique words: {len(counts)}")
print("\nTop 15 words:")
for word, count in top_n(counts, 15):
    print(f"  {word}\t{count}")

Unique words: 96

Top 15 words:
  headphones	3
  delivery	3
  for	3
  was	3
  customer	3
  excellent	2
  quality	2
  fast	2
  after	2
  poor	2
  t	2
  shirt	2
  shoes	2
  packaging	2
  blender	2


**Checkpoint:** You should see words like `delivery`, `headphones`, and `customer` in the top results.

Next we rewrite this logic as scripts Hadoop can run at cluster scale.

---
## Step 3 — Mapper and reducer scripts

**Hadoop Streaming** runs separate Python programs connected by stdin/stdout:

```mermaid
sequenceDiagram
  participant IN as HDFS Input
  participant MAP as Mapper
  participant SH as Shuffle
  participant RED as Reducer
  participant OUT as HDFS Output
  IN->>MAP: line via stdin
  MAP->>SH: key TAB value
  SH->>RED: sorted pairs
  RED->>OUT: final key TAB value
```

| Script | Location | Role |
|--------|----------|------|
| Mapper | `mapreduce/streaming/wordcount_mapper.py` | Emit `(word, 1)` per token |
| Reducer | `mapreduce/streaming/wordcount_reducer.py` | Sum counts per word |

> The Docker Hadoop image uses **Python 2.7**. Scripts are compatible with both Python 2.7 (cluster) and Python 3 (your laptop).

In [18]:
print("=== MAPPER ===")
print(Path("mapreduce/streaming/wordcount_mapper.py").read_text())
print("=== REDUCER ===")
print(Path("mapreduce/streaming/wordcount_reducer.py").read_text())

=== MAPPER ===
#!/usr/bin/env python
"""Map phase: emit (word, 1) for each token in a line of text."""
from __future__ import print_function

import re
import sys

WORD = re.compile(r"[a-zA-Z]+")


def main():
    for line in sys.stdin:
        for word in WORD.findall(line.lower()):
            print("{0}\t{1}".format(word, 1))


if __name__ == "__main__":
    main()

=== REDUCER ===
#!/usr/bin/env python
"""Reduce phase: sum counts for each word key."""
from __future__ import print_function

import sys


def main():
    current_word = None
    current_count = 0

    for line in sys.stdin:
        line = line.strip()
        if not line:
            continue

        word, count = line.split("\t", 1)
        count = int(count)

        if current_word == word:
            current_count += count
        else:
            if current_word is not None:
                print("{0}\t{1}".format(current_word, current_count))
            current_word = word
            current_count = count

 

### Step 3b — Test the Unix pipeline (mini-Hadoop)

This is exactly what Hadoop Streaming does. The `sort` command plays the **shuffle** step:

```
input file  →  mapper  →  sort  →  reducer  →  results
```

In [19]:
import subprocess

sample = reviews_path.read_text(encoding="utf-8")

mapped = subprocess.run(
    ["python3", "mapreduce/streaming/wordcount_mapper.py"],
    input=sample, capture_output=True, text=True, check=True,
)
shuffled = subprocess.run(
    ["sort"], input=mapped.stdout, capture_output=True, text=True, check=True,
)
reduced = subprocess.run(
    ["python3", "mapreduce/streaming/wordcount_reducer.py"],
    input=shuffled.stdout, capture_output=True, text=True, check=True,
)

print("Top 10 from local pipeline:")
rows = sorted(
    reduced.stdout.strip().splitlines(),
    key=lambda r: int(r.split("\t")[1]), reverse=True,
)
for line in rows[:10]:
    print(line)

Top 10 from local pipeline:
customer	3
delivery	3
for	3
headphones	3
was	3
after	2
blender	2
excellent	2
fast	2
is	2


**Checkpoint:** Results should match Step 2. If not, fix the scripts before continuing.

---
## Step 4 — Verify the Hadoop cluster

All 7 containers must be **healthy** and 3 YARN nodes **RUNNING** before you submit jobs.

In [20]:
!docker compose ps

NAME              IMAGE                          COMMAND                  SERVICE           CREATED          STATUS                    PORTS
historyserver     neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   historyserver     48 minutes ago   Up 48 minutes (healthy)   0.0.0.0:19888->19888/tcp, [::]:19888->19888/tcp
namenode          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   namenode          25 minutes ago   Up 25 minutes (healthy)   0.0.0.0:9870->9870/tcp, [::]:9870->9870/tcp, 0.0.0.0:9900->9000/tcp, [::]:9900->9000/tcp
proxyserver       neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   proxyserver       48 minutes ago   Up 48 minutes (healthy)   0.0.0.0:9099->9099/tcp, [::]:9099->9099/tcp
resourcemanager   neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   resourcemanager   25 minutes ago   Up 25 minutes (healthy)   0.0.0.0:8088->8088/tcp, [::]:8088->8088/tcp
worker-1          neshkeev/hadoop:3.3.6-jdk-11   "/tini -- /bin/entry…"   worker-1          25

In [21]:
!docker exec resourcemanager yarn node -list

2026-07-20 12:24:18,656 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:18,685 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
Total Nodes:3
         Node-Id	     Node-State	Node-Http-Address	Number-of-Running-Containers
  worker-3:43151	        RUNNING	   worker-3:38042	                           0
  worker-2:36341	        RUNNING	   worker-2:28042	                           0
  worker-1:43503	        RUNNING	   worker-1:18042	                           0


**Checkpoint:** 7 healthy containers, 3 YARN nodes in `RUNNING` state.

If not, go back to [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) and restart the cluster.

---
## Step 5 — Upload input data to HDFS

MapReduce reads from **HDFS**, not your laptop filesystem.

In [22]:
!docker exec namenode hdfs dfs -mkdir -p /shopstream/raw/reviews
!docker exec namenode hdfs dfs -mkdir -p /shopstream/processed

!docker cp data/ecommerce/product_reviews.txt namenode:/tmp/product_reviews.txt
!docker exec namenode hdfs dfs -put -f /tmp/product_reviews.txt /shopstream/raw/reviews/product_reviews.txt

!docker exec namenode hdfs dfs -ls /shopstream/raw/reviews
!echo "--- Preview ---"
!docker exec namenode hdfs dfs -cat /shopstream/raw/reviews/product_reviews.txt | head -5

2026-07-20 12:24:19,488 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:20,435 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Successfully copied 2.56kB to namenode:/tmp/product_reviews.txt
2026-07-20 12:24:21,690 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:22,719 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Found 1 items
-rw-r--r--   2 hadoop supergroup        808 2026-07-20 12:24 /shopstream/raw/reviews/product_reviews.txt
--- Preview ---
2026-07-20 12:24:23,873 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
Great headphones excellent soun

**Checkpoint:** File appears at `/shopstream/raw/reviews/product_reviews.txt`.

---
## Step 6 — Deploy and submit your MapReduce job

```mermaid
flowchart TB
  A[Laptop: mapper.py + reducer.py] -->|docker cp| B[namenode container]
  C[HDFS input file] --> D[hadoop-streaming JAR]
  B --> D
  D -->|YARN schedules| E[Map tasks on workers]
  E --> F[Shuffle]
  F --> G[Reduce tasks]
  G --> H[HDFS output part-r-*]
  D --> I[Monitor: localhost:8088]
  G --> J[History: localhost:19888]
```

1. Deploy scripts into the cluster
2. Delete any previous output directory
3. Submit via `hadoop-streaming` JAR
4. Watch progress at http://localhost:8088/cluster/apps

In [23]:
from mapreduce.job_helpers import deploy_scripts, submit_streaming_job, extract_tracking_url

STREAMING_DIR = "mapreduce/streaming"
INPUT_PATH = "/shopstream/raw/reviews/product_reviews.txt"
OUTPUT_PATH = "/shopstream/processed/python_wordcount"

deploy_scripts(STREAMING_DIR, ["wordcount_mapper.py", "wordcount_reducer.py"])

print("Submitting wordcount job — watch http://localhost:8088/cluster/apps")
job_log = submit_streaming_job(
    mapper="wordcount_mapper.py",
    reducer="wordcount_reducer.py",
    input_path=INPUT_PATH,
    output_path=OUTPUT_PATH,
)

tracking_url = extract_tracking_url(job_log)
if tracking_url:
    print("Tracking URL:", tracking_url.replace("proxyserver", "localhost"))

Deployed: wordcount_mapper.py, wordcount_reducer.py
Submitting wordcount job — watch http://localhost:8088/cluster/apps
packageJobJar: [/tmp/hadoop-unjar18063395239375237806/] [] /tmp/streamjob8861383326726977174.jar tmpDir=null
2026-07-20 12:24:25,602 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:25,839 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
2026-07-20 12:24:25,895 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
2026-07-20 12:24:25,978 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784548709381_0001
2026-07-20 12:24:26,591 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-20 12:24:26,632 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-20 12:24:26,706 IN

**Checkpoint:** Log ends with `Job ... completed successfully` and progress lines:

```
map 0% reduce 0%  →  map 100% reduce 0%  →  map 100% reduce 100%
```

In [24]:
from mapreduce.job_helpers import read_hdfs_output

print("Top 15 word counts from HDFS:")
read_hdfs_output(OUTPUT_PATH, top_n=15)

Top 15 word counts from HDFS:
was	3
headphones	3
for	3
delivery	3
customer	3
t	2
shoes	2
shirt	2
recommend	2
quality	2
poor	2
packaging	2
moisturizer	2
is	2
fast	2



'was\t3\nheadphones\t3\nfor\t3\ndelivery\t3\ncustomer\t3\nt\t2\nshoes\t2\nshirt\t2\nrecommend\t2\nquality\t2\npoor\t2\npackaging\t2\nmoisturizer\t2\nis\t2\nfast\t2\n'

**Checkpoint:** Results similar to Step 3b. Your Python MapReduce job ran on the cluster.

---
## Step 7 — Monitor MapReduce jobs

```mermaid
flowchart LR
  SUB[Submit job] --> YARN[YARN UI :8088]
  YARN --> RUN[RUNNING state]
  RUN --> PROXY[Proxy :9099]
  RUN --> DONE[FINISHED state]
  DONE --> HIST[History :19888]
  DONE --> HDFS[Read HDFS output]
```

In [25]:
from mapreduce.job_helpers import list_yarn_applications, get_latest_application_id, application_status

print("=== RUNNING ===")
list_yarn_applications("RUNNING")

print("\n=== FINISHED ===")
list_yarn_applications("FINISHED")

=== RUNNING ===
Total number of applications (application-types: [], states: [RUNNING] and tags: []):0
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL


=== FINISHED ===
Total number of applications (application-types: [], states: [FINISHED] and tags: []):1
                Application-Id	    Application-Name	    Application-Type	      User	     Queue	             State	       Final-State	       Progress	                       Tracking-URL
application_1784548709381_0001	streamjob8861383326726977174.jar	           MAPREDUCE	    hadoop	   default	          FINISHED	         SUCCEEDED	           100%	http://historyserver:19888/jobhistory/job/job_1784548709381_0001



'Total number of applications (application-types: [], states: [FINISHED] and tags: []):1\n                Application-Id\t    Application-Name\t    Application-Type\t      User\t     Queue\t             State\t       Final-State\t       Progress\t                       Tracking-URL\napplication_1784548709381_0001\tstreamjob8861383326726977174.jar\t           MAPREDUCE\t    hadoop\t   default\t          FINISHED\t         SUCCEEDED\t           100%\thttp://historyserver:19888/jobhistory/job/job_1784548709381_0001\n'

In [26]:
app_id = get_latest_application_id("FINISHED")
if app_id:
    print(f"Latest finished application: {app_id}")
    application_status(app_id)
else:
    print("No finished applications found yet.")

Latest finished application: application_1784548709381_0001
Application Report : 
	Application-Id : application_1784548709381_0001
	Application-Name : streamjob8861383326726977174.jar
	Application-Type : MAPREDUCE
	User : hadoop
	Queue : default
	Application Priority : 0
	Start-Time : 1784550266808
	Finish-Time : 1784550277081
	Progress : 100%
	State : FINISHED
	Final-State : SUCCEEDED
	Tracking-URL : http://historyserver:19888/jobhistory/job/job_1784548709381_0001
	RPC Port : 38039
	AM Host : worker-3
	Aggregate Resource Allocation : 34060 MB-seconds, 17 vcore-seconds
	Aggregate Resource Preempted : 0 MB-seconds, 0 vcore-seconds
	Log Aggregation Status : DISABLED
	Diagnostics : 
	Unmanaged Application : false
	Application Node Label Expression : <Not set>
	AM container Node Label Expression : <DEFAULT_PARTITION>
	TimeoutType : LIFETIME	ExpiryTime : UNLIMITED	RemainingTime : -1seconds




### Web UI monitoring

| Step | URL | What to check |
|------|-----|---------------|
| 1 | http://localhost:8088/cluster/apps | Application state |
| 2 | Click app → Tracking URL | Map/reduce progress |
| 3 | http://localhost:19888/jobhistory | Counters for finished jobs |
| 4 | http://localhost:9870 | Browse `/shopstream/processed/` |

### Key counters

| Counter | Meaning |
|---------|--------|
| `Map input records` | Lines read from HDFS |
| `Map output records` | Pairs emitted by mapper |
| `Reduce input groups` | Unique keys after shuffle |
| `Launched map tasks` | Parallel map containers |

In [27]:
!docker exec historyserver mapred job -list all 2>/dev/null | tail -5 || echo "Open http://localhost:19888/jobhistory for the full list"

Total jobs:1
                  JobId	             JobName	     State	     StartTime	    UserName	       Queue	  Priority	 UsedContainers	 RsvdContainers	 UsedMem	 RsvdMem	 NeededMem	   AM info
 job_1784548709381_0001	streamjob88613833267	 SUCCEEDED	 1784550266808	      hadoop	     default	   DEFAULT	              1	              0	   2048M	      0M	     2048M	http://localhost:9099/proxy/application_1784548709381_0001/


---
## Step 8 — Compare Python vs built-in Java wordcount

Hadoop ships a Java wordcount example. Same MapReduce pattern — different language.

In [28]:
!docker exec namenode hdfs dfs -rm -r -f /shopstream/processed/review_wordcount_java 2>/dev/null || true

!docker exec namenode bash -lc \
  'hadoop jar $(ls $HADOOP_HOME/share/hadoop/mapreduce/hadoop-mapreduce-examples-*.jar | head -1) wordcount /shopstream/raw/reviews/product_reviews.txt /shopstream/processed/review_wordcount_java'

2026-07-20 12:24:44,684 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:44,913 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
2026-07-20 12:24:45,038 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784548709381_0002
2026-07-20 12:24:45,136 INFO input.FileInputFormat: Total input files to process : 1
2026-07-20 12:24:45,171 INFO mapreduce.JobSubmitter: number of splits:1
2026-07-20 12:24:45,251 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784548709381_0002
2026-07-20 12:24:45,251 INFO mapreduce.JobSubmitter: Executing with tokens: []
2026-07-20 12:24:45,314 INFO conf.Configuration: resource-types.xml not found
2026-07-20 12:24:45,314 INFO resource.ResourceUtils: Unable to find 'resource-types.xml'.
2026-07-20 12:24:45,335 INFO impl.YarnClien

In [29]:
print("=== Python Streaming (top 5) ===")
!docker exec namenode bash -lc "hdfs dfs -cat /shopstream/processed/python_wordcount/part-* | sort -t\$'\\t' -k2 -nr | head -5"

print("\n=== Java examples (top 5) ===")
!docker exec namenode bash -lc "hdfs dfs -cat /shopstream/processed/review_wordcount_java/part-* | sort -t\$'\\t' -k2 -nr | head -5"

=== Python Streaming (top 5) ===
2026-07-20 12:24:56,185 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
was	3
headphones	3
for	3
delivery	3
customer	3

=== Java examples (top 5) ===
2026-07-20 12:24:57,162 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
was	3
for	3
customer	3
recommend	2
quality	2


**Checkpoint:** Both outputs show similar top words — your Python job matches the built-in Java example.

---
## Step 9 — Second job: sentiment keyword counts

**Question:** How many reviews contain positive vs negative keywords?

| Script | Role |
|--------|------|
| `sentiment_mapper.py` | Emit `positive\t1` or `negative\t1` when keywords match |
| `sentiment_reducer.py` | Sum counts per category |

In [30]:
SENTIMENT_OUTPUT = "/shopstream/processed/review_sentiment"

deploy_scripts(STREAMING_DIR, ["sentiment_mapper.py", "sentiment_reducer.py"])

submit_streaming_job(
    mapper="sentiment_mapper.py",
    reducer="sentiment_reducer.py",
    input_path=INPUT_PATH,
    output_path=SENTIMENT_OUTPUT,
)

print("Sentiment results:")
read_hdfs_output(SENTIMENT_OUTPUT, top_n=10)

Deployed: sentiment_mapper.py, sentiment_reducer.py
packageJobJar: [/tmp/hadoop-unjar10610174165034467548/] [] /tmp/streamjob9932923415367827217.jar tmpDir=null
2026-07-20 12:24:58,997 WARN util.NativeCodeLoader: Unable to load native-hadoop library for your platform... using builtin-java classes where applicable
2026-07-20 12:24:59,247 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
2026-07-20 12:24:59,309 INFO client.DefaultNoHARMFailoverProxyProvider: Connecting to ResourceManager at resourcemanager/172.22.0.4:8032
2026-07-20 12:24:59,391 INFO mapreduce.JobResourceUploader: Disabling Erasure Coding for path: /tmp/hadoop-yarn/staging/hadoop/.staging/job_1784548709381_0003
2026-07-20 12:24:59,524 INFO mapred.FileInputFormat: Total input files to process : 1
2026-07-20 12:24:59,550 INFO mapreduce.JobSubmitter: number of splits:2
2026-07-20 12:24:59,623 INFO mapreduce.JobSubmitter: Submitting tokens for job: job_1784548709

'positive\t10\nnegative\t6\n'

**Expected:** Two lines — `positive` and `negative` counts. One review can match both.

---
## Step 10 — Job lifecycle summary

```
DEVELOP (laptop)              DEPLOY (cluster)
────────────────              ────────────────
Write mapper.py               Start Hadoop cluster
Write reducer.py              Upload input → HDFS
Test: cat | map | sort | red  docker cp scripts → namenode
Fix until correct             hadoop jar streaming.jar ...
                              Monitor 8088 / 19888
                              Read part-r-* from HDFS
```

### Common errors

| Error | Fix |
|-------|-----|
| `Output directory already exists` | Delete output path before re-run |
| `Connection refused` | Wait for cluster healthy |
| Empty reducer output | Check mapper emits `key\tvalue` |
| Python error in cluster | Use provided scripts (Python 2.7 compatible) |

---
## Step 11 — Map to AWS

| Local (this lab) | AWS equivalent |
|------------------|----------------|
| HDFS paths | Amazon S3 |
| `hadoop-streaming` JAR | Amazon EMR step |
| YARN UI (:8088) | EMR console |
| History Server (:19888) | EMR logs in S3 + CloudWatch |

---

## Reference

| Resource | Path |
|----------|------|
| Written guide | [MAPREDUCE-STUDENT-GUIDE.md](./MAPREDUCE-STUDENT-GUIDE.md) |
| Module code | [mapreduce/](./mapreduce/) |
| Cluster setup | [HADOOP-STUDENT-GUIDE.md](./HADOOP-STUDENT-GUIDE.md) |
| Full cluster tour | [Hadoop-Local-Cluster-Guide.ipynb](./Hadoop-Local-Cluster-Guide.ipynb) |